In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_natujur")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_natujur (
  natujur_id INT,
  natujur_descripcion STRING
)
USING DELTA
""")

In [0]:
import pandas as pd

df = spark.table("workspace.silver.predios_registro").toPandas()

In [0]:
from pyspark.sql import functions as F

dim = (
    df[[
        "natujur_id",
        "natujur_descripcion"
    ]]
    .dropna(subset=["natujur_id"])
    .drop_duplicates(subset=["natujur_id"])
    .sort_values("natujur_descripcion", ignore_index=True)
)

df_spark = spark.createDataFrame(dim)

df_spark = df_spark.withColumn("natujur_id", F.col("natujur_id").cast("int"))

In [0]:

# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_natujur")